# AI for Molecules — Hands-on Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Corteswain/AI4molecules-tutorial/blob/main/notebooks/AI4molecules_tutorial.ipynb)

In this 75-minute tutorial you'll build a molecular property predictor from scratch. First you'll
run a standard data-cleaning pipeline, then make four modeling decisions in sequence:

0. **Data cleaning** — before anything else, put the raw SMILES through a standard curation
   pipeline (removing invalid entries, canonicalizing, stripping salts, removing stereochemistry,
   removing duplicates). Comes first because every later step assumes clean, consistent SMILES.
1. **Dataset splitting** — how do we split data into train, validation, and test sets so our
   performance estimate is trustworthy? Comes next because it only touches the SMILES and the
   target, not any model or features. `test` is a true holdout: nothing else in the tutorial
   ever looks at it until the very end.
2. **Model choice** — a classical ML model (Random Forest / XGBoost) on hand-crafted features, or
   a graph neural network (Chemprop) that learns its own representation? This determines whether
   the next step even applies.
3. **Representation** — how do we turn a molecule (a SMILES string) into numbers a model can use?
   (Only relevant if you didn't pick Chemprop — more on that below.)
4. **Train & evaluate** — put it all together and see how it did.

At each of the four modeling-decision steps (1 through 4), a few options are already implemented
as functions in `utils/`. **You choose which one to plug in**, run it, and then we compare notes
as a group before moving to the next step.

**Task:** predict aqueous solubility (logS) from molecular structure, using the [ESOL dataset](https://pubs.acs.org/doi/10.1021/ci034243x) (1,128 small molecules).

**Agenda (75 min)**

| Time | Section |
|---|---|
| 0–10 min | Setup |
| 11–20 min | Step 0: Data cleaning |
| 21–30 min | Step 1: Splitting |
| 31–35 min | Step 2: Model choice |
| 36–50 min | Step 3: Representation |
| 51–60 min | Step 4: Train & evaluate |
| 61–75 min | Wrap-up & discussion |


## Setup

Run the cell below once. On Colab it installs the needed packages and pulls in the `utils/` code and dataset from the tutorial repo (takes ~1-2 minutes).

In [1]:
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q rdkit scikit-learn xgboost chemprop pandas matplotlib
    if not os.path.exists("AI4molecules-tutorial"):
        !git clone --quiet https://github.com/Corteswain/AI4molecules-tutorial.git
    %cd AI4molecules-tutorial
else:
    # This notebook always lives at <repo_root>/notebooks/AI4molecules_tutorial.ipynb, so
    # the repo root is exactly one directory up from here. VS Code's Jupyter extension
    # exposes the notebook's own file path as __vsc_ipynb_file__ — we use that (not the
    # kernel's working directory, which VS Code does not set to the notebook's folder).
    os.chdir(Path(__vsc_ipynb_file__).resolve().parent.parent)

sys.path.insert(0, ".")
print("Setup done. Running in Colab:", IN_COLAB, "| working directory:", os.getcwd())

Setup done. Running in Colab: False | working directory: /home/ricbec/projects/AI4molecules-tutorial


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.representations import REPRESENTATIONS
from utils.splitting import SPLITTERS, random_split, scaffold_split, kmeans_split, butina_split
from utils.models import MODEL_RUNNERS, run_chemprop
from utils.cleaning import remove_invalid_smiles, canonicalize_smiles, remove_salts, remove_stereochemistry, remove_duplicates
from utils.viz import low_dimensional_representation, nearest_neighbor_similarity, TRAIN_COLOR, VAL_COLOR, TEST_COLOR

TARGET = "measured_log_solubility_mol_per_L"
df = pd.read_csv("data/esol.csv")
print(f"{len(df)} molecules")
df.head()

## Step 0 — Data cleaning

Before splitting, choosing a model, or featurizing anything, put the raw SMILES through a
standard cheminformatics curation pipeline. This comes first because every later step assumes
the SMILES are valid, consistently formatted, and free of things (salts, stereochemistry) that
would otherwise need to be decided on implicitly. Five functions are implemented in
`utils/cleaning.py`, each doing one job:

- **`remove_invalid_smiles`** — drops rows RDKit can't parse at all. Has to run first: every
  other step assumes a valid molecule.
- **`canonicalize_smiles`** — rewrites every SMILES into RDKit's canonical form, so two ways of
  writing the same molecule aren't treated as different molecules downstream.
- **`remove_salts`** — strips counterions/salt fragments (e.g. "CCN.Cl" → "CCN"), keeping only
  the parent molecule.
- **`remove_stereochemistry`** — removes chiral tags and E/Z bond stereo. A real modeling
  choice, not just cleanup — see the discussion below.
- **`remove_duplicates`** — drops duplicate molecules, keeping the first occurrence. Runs last,
  since canonicalizing/desalting/destereo-ing are exactly what can turn molecules that looked
  different into exact duplicates.

Run the cell below to apply them one at a time, in order — each one prints how many molecules it
actually changed.

In [ ]:
print(f"Starting with {len(df)} molecules")

df = remove_invalid_smiles(df)
df = canonicalize_smiles(df)
df = remove_salts(df)
df = remove_stereochemistry(df)
df = remove_duplicates(df, target_col=TARGET)

print(f"Finished with {len(df)} molecules")

**Discuss:**
- This dataset came out with 0 invalid SMILES and 0 salts — is that surprising for a
  widely-used benchmark like ESOL? What kind of dataset (e.g. scraped from a database, or
  submitted by many different labs) would you expect this pipeline to matter much more for?
- `canonicalize_smiles` changed most of the dataset (the same molecule can be written many
  different ways). Why does that matter for machine learning, even though it doesn't change
  what molecule is represented?
- `remove_stereochemistry` changed a handful of molecules. What could go wrong if you're
  predicting a property that actually depends on stereochemistry (e.g. drug activity, taste,
  smell) and you strip it anyway?
- `remove_duplicates` reported some duplicate groups that *disagree* on the target value.
  Look at one of them (search the original data for its SMILES) — is the disagreement
  measurement noise, or did an earlier step (hint: which one?) make two genuinely different
  molecules look identical? What would you do differently if you found this before running
  `remove_stereochemistry`?

In [ ]:
df[TARGET].hist(bins=30, color=plt.cm.viridis(0.5))
plt.xlabel("measured log solubility (mol/L)")
plt.ylabel("count")
plt.title("ESOL target distribution")
plt.show()

## Step 1 — Dataset splitting

We start here because splitting only touches the SMILES and the target — it doesn't depend on a
model or a representation, so there's no reason to decide those first. Every split function
returns **three** sets, not two: `train`, `val`, and `test`. `test` is a *true holdout* — nothing
in this tutorial trains on it, tunes anything using it, or picks a checkpoint based on it; it's
touched exactly once, at final evaluation in Step 4. `val` is what's used instead for anything
that needs feedback along the way (Chemprop's early stopping, or your own informal checks). If
you ever pick a model, representation, or hyperparameter because it looked good on `test`, `test`
quietly stops being a true holdout — it's just become another validation set.

How we split molecules changes what our evaluation actually measures. Four options are
implemented in `utils/splitting.py`, each targeting ~15% val / ~15% test (train gets the rest):

- **`random`** — a plain i.i.d. shuffle-and-split. No relationship between molecules is considered.
- **`scaffold`** — molecules are grouped by [Bemis-Murcko scaffold](https://en.wikipedia.org/wiki/Bemis%E2%80%93Murcko_scaffold) (their shared ring/core structure); whole scaffold groups go to train, val, or test, so near-identical molecules can't leak across the split.
- **`kmeans`** — molecules are clustered by *Euclidean* distance between fingerprint vectors (KMeans), and whole clusters are held out for val/test.
- **`butina`** — molecules are clustered by *Tanimoto* similarity between fingerprints ([Butina clustering](https://www.rdkit.org/docs/Cookbook.html#clustering-molecules), the standard cheminformatics notion of "similar molecule"), and whole clusters are held out for val/test.

`kmeans` and `butina` both aim to test extrapolation to structurally distinct
regions of chemical space, but they can disagree with each other: two molecules "close" by
Euclidean distance aren't guaranteed to be "close" by Tanimoto similarity, especially near cluster
boundaries.

Rather than picking one blind, run the cell below to compute **all four splits** — each one is
called explicitly, with a short message when it's done, so you can see exactly which computation
is happening and how long each one takes.

In [ ]:
print("Running random_split...")
random_train_idx, random_val_idx, random_test_idx = random_split(df, val_size=0.15, test_size=0.15, seed=42)
print(f"  done — {len(random_train_idx)} train / {len(random_val_idx)} val / {len(random_test_idx)} test")

print("Running scaffold_split...")
scaffold_train_idx, scaffold_val_idx, scaffold_test_idx = scaffold_split(df, val_size=0.15, test_size=0.15, seed=42)
print(f"  done — {len(scaffold_train_idx)} train / {len(scaffold_val_idx)} val / {len(scaffold_test_idx)} test")

print("Running kmeans_split...")
kmeans_train_idx, kmeans_val_idx, kmeans_test_idx = kmeans_split(df, val_size=0.15, test_size=0.15, seed=42)
print(f"  done — {len(kmeans_train_idx)} train / {len(kmeans_val_idx)} val / {len(kmeans_test_idx)} test")

print("Running butina_split...")
butina_train_idx, butina_val_idx, butina_test_idx = butina_split(df, val_size=0.15, test_size=0.15, seed=42)
print(f"  done — {len(butina_train_idx)} train / {len(butina_val_idx)} val / {len(butina_test_idx)} test")

splits = {
    "random": (random_train_idx, random_val_idx, random_test_idx),
    "scaffold": (scaffold_train_idx, scaffold_val_idx, scaffold_test_idx),
    "kmeans": (kmeans_train_idx, kmeans_val_idx, kmeans_test_idx),
    "butina": (butina_train_idx, butina_val_idx, butina_test_idx),
}


nn_sim_stats = nearest_neighbor_similarity(df, splits, k=5)

Before picking one to focus on, compare all four at once: for each split, how Tanimoto-similar
is each test molecule to its 5th-nearest neighbor in that split's training set? A high similarity
means test molecules typically have several close analogs already in train (easy interpolation);
a low similarity means the split is testing genuine extrapolation to unfamiliar chemistry.

Now pick one of the four to inspect more closely.

In [ ]:
# ---- YOUR CHOICE ----
SPLIT_METHOD = "scaffold"  # options: "random", "scaffold", "kmeans", "butina"
# ----------------------

train_idx, val_idx, test_idx = splits[SPLIT_METHOD]
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
y_train = df[TARGET].values[train_idx]
y_val = df[TARGET].values[val_idx]
y_test = df[TARGET].values[test_idx]

print(f"Using {SPLIT_METHOD}: {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")

plt.hist(y_train, bins=30, alpha=0.6, label="train", density=True, color=TRAIN_COLOR)
plt.hist(y_val, bins=30, alpha=0.6, label="val", density=True, color=VAL_COLOR)
plt.hist(y_test, bins=30, alpha=0.6, label="test", density=True, color=TEST_COLOR)
plt.xlabel("measured log solubility (mol/L)")
plt.ylabel("density")
plt.legend()
plt.title(f"Target distribution: {SPLIT_METHOD} split")
plt.show()

split_stats = low_dimensional_representation(df, train_idx, val_idx, test_idx)

**Discuss:**
- Do train, val, and test look like they come from the same distribution (histogram) and occupy the same regions of chemical space (t-SNE/PCA)?
- What fraction of Butina clusters mix test with train/val molecules? What would you expect that number to be for each split method, and does the plot match?
- Try switching `SPLIT_METHOD` to `kmeans` and re-run from there — both it and `butina` "hold out whole clusters," so why doesn't `kmeans` also drive the mixed-cluster fraction to 0%?
- With `scaffold` or either `cluster_*` method, are you testing *interpolation* (molecules similar to what the model has seen) or *extrapolation* (genuinely new chemistry)?
- Which split would you trust more as an estimate of performance on molecules made in a lab next year?

## Step 2 — Model choice

Now that the split is fixed, decide which model to train. This choice determines whether
hand-crafting features (Step 3) is even part of the workflow. Three options are implemented in
`utils/models.py`:

- **`random_forest`** / **`xgboost`** — classical ML models. They need a fixed-size feature vector
  per molecule, which means *you* have to decide how to turn a molecule into numbers (Step 3).
- **`chemprop`** — a message-passing graph neural network (MPNN) that learns its own representation
  directly from the molecular graph, end to end, while training. It doesn't need a hand-crafted
  feature vector from Step 3 to get started — but it isn't feature-blind either; see Step 3 for
  what it uses by default and how to extend it.

Pick one below. (Chemprop takes ~20-40 seconds to train, later in Step 4.)

In [ ]:
# ---- YOUR CHOICE ----
MODEL = "random_forest"  # options: "random_forest", "xgboost", "chemprop"
# ----------------------

print(f"Selected model: {MODEL}")
if MODEL == "chemprop":
    print("Chemprop learns its own representation from the molecular graph — Step 3's functions won't apply, but see Step 3 for its default features.")

**Discuss:**
- Random Forest and XGBoost are trained on a fixed feature vector — what does that assume about the features you'll pick in Step 3?
- Chemprop trains directly on the molecular graph. What does it need *more* of (data, compute, time) to make up for not having hand-crafted features?
- If you only had 50 molecules instead of ~1,000, would your choice of model change?

## Step 3 — Representation

For `random_forest` / `xgboost`, pick how to turn a molecule into numbers. Three options are
implemented in `utils/representations.py`:

- **`morgan`** — Morgan (circular) fingerprints: a 2048-bit vector encoding which local substructures are present.
- **`maccs`** — MACCS keys: a fixed 166-bit vector, each bit a specific, human-named structural pattern (e.g. "has a ring of size 6").
- **`rdkit_descriptors`** — 217 global physicochemical descriptors: molecular weight, LogP, TPSA, H-bond donors/acceptors, rotatable bonds, ring counts...

**If you picked `chemprop` in Step 2, these three functions don't apply to it** — but that doesn't
mean representation is irrelevant to Chemprop. It already makes a representation choice by
default (a fixed set of per-atom and per-bond features it builds the molecular graph from), and,
like the options above, that default can be swapped out or extended with your own hand-crafted
features. See the [Chemprop v2 documentation](https://chemprop.readthedocs.io/en/latest/) for the
full set of options (`--molecule-featurizers`, `--descriptors-path`, `--atom-features-path`,
`--bond-features-path`, and more) — most are only exposed via the CLI/config, not (yet) wired up
in `utils/models.py`'s `run_chemprop`.

Chemprop v2's **default** per-atom (node) and per-bond (edge) features, verified against the
installed `chemprop==2.3.1` (`chemprop.featurizers.atom.MultiHotAtomFeaturizer.v2()` /
`chemprop.featurizers.bond.MultiHotBondFeaturizer()`):

- **Node (atom) features — 72 values per atom:** atomic number (one-hot, H through Kr plus
  iodine), degree, formal charge, chiral tag, number of attached hydrogens, hybridization
  (s / sp / sp² / sp²d / sp³ / sp³d / sp³d²), aromaticity, and atomic mass.
- **Edge (bond) features — 14 values per bond:** a "null" padding bit, bond type
  (single/double/triple/aromatic), whether it's conjugated, whether it's in a ring, and its
  stereochemistry (E/Z or none).

Pick one below (ignored if `MODEL == "chemprop"`).

In [ ]:
# ---- YOUR CHOICE ----
REPRESENTATION = "morgan"  # options: "morgan", "maccs", "rdkit_descriptors"
# ----------------------

if MODEL == "chemprop":
    print("MODEL = 'chemprop' — it builds its own representation from the graph, so this choice will be ignored in Step 4.")
    X_train = X_val = X_test = None
else:
    featurize = REPRESENTATIONS[REPRESENTATION]
    X_train = featurize(train_df["smiles"].tolist())
    X_val = featurize(val_df["smiles"].tolist())
    X_test = featurize(test_df["smiles"].tolist())
    print(f"{REPRESENTATION}: X_train.shape = {X_train.shape}")

**Discuss:**
- How many numbers describe each molecule here? What structural information might this representation lose?
- Would two molecules a chemist considers very similar end up with similar feature vectors?
- Could you look at this feature vector and explain *why* the model made a given prediction?
- We do not have the time to manually optimise chemprop's features, but what would you like to change if given the opportunity?

## Step 4 — Train & evaluate

Now put your three choices together: train `MODEL` on the `SPLIT_METHOD` split, using the
`REPRESENTATION` features (unless you picked `chemprop`, which trains directly on SMILES).

In [ ]:
if MODEL == "chemprop":
    # Chemprop's "representation learning" is, in effect, its own version of Step 3 —
    # it folds feature-crafting into the model rather than doing it by hand.
    result = run_chemprop(train_df, val_df, test_df, target_col=TARGET, epochs=30)
    label = f"{MODEL} ({SPLIT_METHOD} split, learned representation)"
else:
    result = MODEL_RUNNERS[MODEL](X_train, y_train, X_val, y_val, X_test, y_test)
    label = f"{MODEL} ({SPLIT_METHOD} split, {REPRESENTATION} representation)"

print(label)
print(f"val:  RMSE = {result['val_rmse']:.3f}   R2 = {result['val_r2']:.3f}")
print(f"test: RMSE = {result['rmse']:.3f}   R2 = {result['r2']:.3f}   <- true holdout, the number that counts")

In [ ]:
y_pred = result["y_pred"]
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.scatter(y_test, y_pred, alpha=0.5, color=plt.cm.viridis(0.5))
plt.plot(lims, lims, "k--", linewidth=1)
plt.xlabel("measured")
plt.ylabel("predicted")
plt.title(f"{label} \u2014 test (true holdout)")
plt.show()

**Discuss:**
- How does your RMSE/R² compare to what your neighbor got with a different choice at any step?
- Random Forest / XGBoost only ever see the Step 3 features — if performance is poor, is that the model's fault or the representation's?
- Chemprop folded feature-crafting into the model itself. Did it do better or worse than the hand-crafted features here? (Hint: ESOL's target was historically fit using a formula built from LogP, molecular weight, and aromaticity — some of the exact descriptors in `rdkit_descriptors`.) What are the benefits and drawbacks of letting a model *learn* its own representation instead of hand-designing one?
- Compare your val and test numbers. Are they close? If you (or a neighbor) kept re-running Steps 2-3 with different choices until val looked good, would you still trust the test number as an honest estimate of performance on genuinely new molecules?